# Multi Layer Perceptron - Batch size, weight initialization and optimizer

This notebook experiments with how batch size, weight initialization and optimizer choice influence the classification results.

In [ ]:
import sys

sys.path.append("..")

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
heart_disease = fetch_ucirepo(id=45)

X_raw: pd.DataFrame = heart_disease.data.features
y_raw: pd.Series = (heart_disease.data.targets["num"] > 0).astype(int)

In [ ]:
mask = ~np.asarray(pd.isna(X_raw).any(axis=1))

X = X_raw[mask]
y = y_raw[mask]

In [ ]:
from models.mlp import MultiLayerPerceptron
from models.activation_functions import ReLu
from models.loss_functions import LogLoss
from models.optimizer import SGD, Adam
from models.weight_initializer import NormalInitializer, HeInitializer

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NUM_COLS = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

prep = ColumnTransformer(
    transformers=[
        ("stand", StandardScaler(), NUM_COLS),
    ],
    remainder="passthrough",
)

In [ ]:
from itertools import product

from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y.to_numpy(), test_size=0.2, stratify=y, random_state=42
)


def loss_plateau_stop(window: int = 20, tol: float = 1e-4):
    def criterion(history: dict) -> bool:
        losses = history["loss"]
        if len(losses) < window:
            return False
        recent = losses[-window:]
        return max(recent) - min(recent) < tol

    return criterion


def evaluate_pipe(pipe, X_test, y_test):
    y_pred = pipe.predict(X_test)
    return {
        "accuracy": round(accuracy_score(y_test, y_pred), 2),
        "f1": round(f1_score(y_test, y_pred, zero_division=0), 2),
        "precision": round(precision_score(y_test, y_pred, zero_division=0), 2),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 2),
    }

## Batch size

Batch sizes: `1`, `32`, `64`, `279` (full dataset).

Architectures: `[13]`, `[24, 16]`, `[2, 4, 8, 16, 32]` with and without standardization.

Defaults: ReLU, LogLoss, `NormalInitializer(0.1)`, SGD.

In [ ]:
batch_sizes = [1, 32, 64, 279]
batch_architectures = [
    [13],
    [24, 16],
    [2, 4, 8, 16, 32],
]

batch_results = []

for hidden_layer_sizes, batch_size in product(batch_architectures, batch_sizes):
    pipe = Pipeline(
        [
            ("prep", clone(prep)),
            (
                "clf",
                MultiLayerPerceptron(
                    hidden_layer_sizes=hidden_layer_sizes,
                    activation_function=ReLu(),
                    loss_function=LogLoss(),
                    weight_initializer=NormalInitializer(0.1, random_state=42),
                    optimizer=SGD(),
                    batch_size=batch_size,
                    random_state=42,
                ),
            ),
        ]
    )

    pipe.fit(X_train, y_train)
    metrics = evaluate_pipe(pipe, X_test, y_test)

    print(
        f"hidden_layer_sizes={hidden_layer_sizes}, "
        f"batch_size={batch_size}, "
        f"accuracy={metrics['accuracy']:.2f}, "
        f"f1={metrics['f1']:.2f}, "
        f"precision={metrics['precision']:.2f}, "
        f"recall={metrics['recall']:.2f}"
    )

    pipe.named_steps["clf"].plot_history()
    plt.show()

    batch_results.append(
        {
            "hidden_layer_sizes": str(hidden_layer_sizes),
            "batch_size": batch_size,
            **metrics,
        }
    )

batch_results_df = pd.DataFrame(batch_results)
batch_results_df.sort_values(
    by=["hidden_layer_sizes", "batch_size"],
    ascending=[True, True],
).reset_index(drop=True)

## Weight initialization

Initializers: `NormalInitializer(0.1)`, `NormalInitializer(1)`, `NormalInitializer(10)`, `ZeroInitializer`, `HeInitializer`.

Architectures: `[13]`, `[24, 16]`, `[2, 4, 8, 16, 32]` with and without standardization.

Defaults: ReLU, LogLoss, SGD, `batch_size=32`.

In [ ]:
from models.weight_initializer import ZeroInitializer

initializer_factories = [
    ("Normal(0.1)", lambda: NormalInitializer(0.1, random_state=42)),
    ("Normal(1)", lambda: NormalInitializer(1, random_state=42)),
    ("Normal(10)", lambda: NormalInitializer(10, random_state=42)),
    ("0", lambda: ZeroInitializer()),
    ("He", lambda: HeInitializer(random_state=42)),
]

init_architectures = [
    [13],
    [24, 16],
    [2, 4, 8, 16, 32],
]

init_results = []

for hidden_layer_sizes, (init_name, make_initializer) in product(
    init_architectures, initializer_factories
):
    pipe = Pipeline(
        [
            ("prep", clone(prep)),
            (
                "clf",
                MultiLayerPerceptron(
                    hidden_layer_sizes=hidden_layer_sizes,
                    activation_function=ReLu(),
                    loss_function=LogLoss(),
                    weight_initializer=make_initializer(),
                    optimizer=SGD(),
                    random_state=42,
                ),
            ),
        ]
    )

    pipe.fit(X_train, y_train)
    metrics = evaluate_pipe(pipe, X_test, y_test)

    print(
        f"hidden_layer_sizes={hidden_layer_sizes}, "
        f"initializer={init_name}, "
        f"accuracy={metrics['accuracy']:.2f}, "
        f"f1={metrics['f1']:.2f}, "
        f"precision={metrics['precision']:.2f}, "
        f"recall={metrics['recall']:.2f}"
    )

    pipe.named_steps["clf"].plot_history()
    plt.show()

    init_results.append(
        {
            "hidden_layer_sizes": str(hidden_layer_sizes),
            "initializer": init_name,
            **metrics,
        }
    )

init_results_df = pd.DataFrame(init_results)
init_results_df.sort_values(
    by=["hidden_layer_sizes", "initializer"],
    ascending=[True, True],
).reset_index(drop=True)

## Optimizer

Optimizers: SGD, Adam.

Architectures: `[13, 13, 13]`, `[8]` — only with standardization.

Stop criteria:
- no early stopping, `max_iter=2000`
- plateau stop (last 20 loss values within `1e-2`), `max_iter=100000`

Defaults: ReLU, LogLoss, `NormalInitializer(0.1)`, `batch_size=32`.

In [ ]:
optimizer_factories = [
    ("SGD", SGD),
    ("Adam", Adam),
]

stop_criteria = [
    ("none_2000", None, 2000),
    ("plateau", loss_plateau_stop(window=20, tol=1e-2), 10_000),
]

optimizer_architectures = [
    [13, 13, 13],
    [8],
]

optimizer_results = []

for hidden_layer_sizes, (opt_name, make_optimizer), (
    stop_name,
    stop_criterion,
    max_iter,
) in product(optimizer_architectures, optimizer_factories, stop_criteria):
    pipe = Pipeline(
        [
            ("prep", clone(prep)),
            (
                "clf",
                MultiLayerPerceptron(
                    hidden_layer_sizes=hidden_layer_sizes,
                    activation_function=ReLu(),
                    loss_function=LogLoss(),
                    weight_initializer=NormalInitializer(0.1, random_state=42),
                    optimizer=make_optimizer(),
                    max_iter=max_iter,
                    stop_criterion=stop_criterion,
                    random_state=42,
                ),
            ),
        ]
    )

    pipe.fit(X_train, y_train)
    metrics = evaluate_pipe(pipe, X_test, y_test)
    n_epochs = len(pipe.named_steps["clf"].history["loss"])

    print(
        f"hidden_layer_sizes={hidden_layer_sizes}, "
        f"optimizer={opt_name}, "
        f"stop={stop_name}, "
        f"epochs={n_epochs} | "
        f"accuracy={metrics['accuracy']:.2f}, "
        f"f1={metrics['f1']:.2f}, "
        f"precision={metrics['precision']:.2f}, "
        f"recall={metrics['recall']:.2f}"
    )

    pipe.named_steps["clf"].plot_history()
    plt.show()

    optimizer_results.append(
        {
            "hidden_layer_sizes": str(hidden_layer_sizes),
            "optimizer": opt_name,
            "stop_criterion": stop_name,
            "epochs": n_epochs,
            **metrics,
        }
    )

optimizer_results_df = pd.DataFrame(optimizer_results)
optimizer_results_df.sort_values(
    by=["hidden_layer_sizes", "optimizer", "stop_criterion"],
    ascending=[True, True, True],
).reset_index(drop=True)